# 🔬 LAB DAY 19: XÂY DỰNG HỆ THỐNG GRAPHRAG VỚI EV CORPUS

**Sinh viên:** Nguyễn Quang Anh — Mã SV: 2A202600608  
**Dataset:** US Electric Vehicle Sector Corpus (70 tài liệu)  
**Framework:** NetworkX + Groq LLM + ChromaDB

---

## Pipeline Tổng Quan

```
70 EV Documents
      │
      ▼
  [Step 1] Data Loading & Chunking
      │
      ▼
  [Step 2] Entity & Relation Extraction (Groq LLM)
      │                          │
      │                          ▼
      │                     Triples JSON
      ▼
  [Step 3] Knowledge Graph Construction (NetworkX)
      │
      ├──► Visualization (Matplotlib)
      ▼
  [Step 4] Multi-hop Query Engine (BFS)
      │
      │         [Parallel] Flat RAG (ChromaDB)
      ▼                         │
  [Step 5] Evaluation & Comparison ◄──────┘
```

## 📦 Phần 0: Cài Đặt Dependencies

In [ ]:
# Cài đặt tất cả dependencies (chạy một lần)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])
print('✅ Cài đặt hoàn thành!')

In [ ]:
# Import thư viện chuẩn
import os
import sys
import json
import warnings
from pathlib import Path

# Thêm src/ vào Python path
sys.path.insert(0, str(Path('src')))

warnings.filterwarnings('ignore')

import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from groq import Groq
from IPython.display import Image, display, HTML

# Modules của chúng ta
from data_loader import load_documents, load_chunks
from entity_extractor import extract_all_triples, get_triple_stats
from graph_builder import build_graph, load_graph, visualize_graph, visualize_subgraph, get_graph_stats
from graph_query import query_graphrag, batch_query_graphrag
from flat_rag import FlatRAG
from evaluator import BENCHMARK_QUESTIONS, run_evaluation, print_evaluation_summary, plot_evaluation

print('✅ Tất cả imports thành công!')
print(f'   NetworkX version: {nx.__version__}')
print(f'   Pandas version: {pd.__version__}')

In [ ]:
# ⚠️ CẤU HÌNH API KEY
# Điền GROQ_API_KEY của bạn vào đây (lấy miễn phí tại https://console.groq.com)
# HOẶC đặt biến môi trường: set GROQ_API_KEY=your_key_here

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', 'your_groq_api_key_here')

if GROQ_API_KEY == 'your_groq_api_key_here':
    raise ValueError('⚠️ Vui lòng đặt GROQ_API_KEY! Lấy miễn phí tại https://console.groq.com')

# Test kết nối
groq_client = Groq(api_key=GROQ_API_KEY)
test_resp = groq_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Say "OK" in one word'}],
    max_tokens=5
)
print(f'✅ Groq API kết nối thành công! Response: {test_resp.choices[0].message.content}')

---
## 📂 Phần 1: Load & Khám Phá Dữ Liệu

In [ ]:
# Load tất cả 70 tài liệu
docs = load_documents(dataset_dir='dataset')

# Hiển thị thống kê cơ bản
total_words = sum(len(d['clean_text'].split()) for d in docs)
avg_words = total_words // len(docs)

print(f'\n📊 Dataset Statistics:')
print(f'  Số tài liệu: {len(docs)}')
print(f'  Tổng từ: {total_words:,}')
print(f'  Trung bình mỗi doc: {avg_words:,} từ')
print(f'  Doc lớn nhất: {max(len(d["clean_text"].split()) for d in docs):,} từ')
print(f'  Doc nhỏ nhất: {min(len(d["clean_text"].split()) for d in docs):,} từ')

In [ ]:
# Xem 3 tài liệu đầu
for doc in docs[:3]:
    print(f"\n📄 {doc['doc_id']}: {doc['title'][:80]}")
    print(f"   URL: {doc['url'][:70]}")
    print(f"   Nội dung ({len(doc['clean_text'].split())} từ): {doc['clean_text'][:200]}...")
    print('-' * 80)

In [ ]:
# Chia thành chunks
chunks = load_chunks(docs)
print(f'\n📊 Chunking Results:')
print(f'  Tổng chunks: {len(chunks)}')
print(f'  Chunk đầu tiên (doc_1_0):')
print(f'  {chunks[0]["text"][:400]}...')

---
## 🧠 Phần 2: Trích Xuất Entities & Relations (Indexing)

**Bước 1:** Dùng Groq `llama-3.3-70b-versatile` để đọc từng chunk và extract Knowledge Graph Triples.

**Input:** `"Tesla's sales in the U.S. were down 13.3% year over year in Q1 2024"`  
**Output Triple:** `{subject: "Tesla", relation: "REPORTED_SALES_DECLINE", object: "13.3% YoY in Q1 2024"}`

> **Lưu ý:** Kết quả được cache vào `output/triples.json`. Chạy lại sẽ load từ cache.

In [ ]:
import time

# Extract triples từ tất cả 70 tài liệu
# use_cache=True: nếu đã extract trước đó, load từ cache
start_time = time.time()

triples = extract_all_triples(
    chunks=chunks,
    groq_api_key=GROQ_API_KEY,
    use_cache=True,  # Đổi thành False để extract lại từ đầu
)

extraction_time = time.time() - start_time
print(f'\n⏱️  Extraction time: {extraction_time:.1f}s')

In [ ]:
# Thống kê triples
stats = get_triple_stats(triples)

print(f'\n📊 Triple Statistics:')
print(f'  Tổng triples: {stats["total_triples"]:,}')
print(f'  Unique entities: {stats["unique_entities"]:,}')
print(f'  Unique relations: {stats["unique_relations"]:,}')

print(f'\n🔝 Top 10 entities (most mentioned):')
for entity, count in stats['top_entities'][:10]:
    print(f'   {entity}: {count} mentions')

print(f'\n🔗 Top 10 relations:')
for rel, count in stats['top_relations']:
    print(f'   {rel}: {count} occurrences')

In [ ]:
# Xem ví dụ 10 triples đầu tiên
print('\n📋 Ví dụ 10 triples đầu tiên:')
print('-' * 80)
for i, t in enumerate(triples[:10], 1):
    print(f'{i:2d}. ({t["subject"]}) --[{t["relation"]}]--> ({t["object"]})')
    print(f'     Source: {t.get("source_doc", "N/A")}')

---
## 🕸️ Phần 3: Xây Dựng Knowledge Graph (NetworkX)

**Bước 2:** Chuyển triples thành đồ thị có hướng (DiGraph) với:
- **Nodes:** Entities (công ty, người, khái niệm, địa điểm...)
- **Edges:** Relations giữa các entities
- **Deduplication:** Gộp các entity tương tự (fuzzy matching)

In [ ]:
# Build Knowledge Graph
G = build_graph(triples)

# Graph statistics
graph_stats = get_graph_stats(G)
print(f'\n📊 Knowledge Graph Statistics:')
print(f'  Số nodes (entities): {graph_stats["num_nodes"]:,}')
print(f'  Số edges (relations): {graph_stats["num_edges"]:,}')
print(f'  Weakly connected components: {graph_stats["num_components"]}')
print(f'  Avg degree: {graph_stats["avg_degree"]:.2f}')

print(f'\n🏷️  Entity type distribution:')
for etype, count in sorted(graph_stats['entity_types'].items(), key=lambda x: -x[1]):
    bar = '█' * (count // 2)
    print(f'  {etype:10s}: {count:4d} {bar}')

print(f'\n🔝 Top 10 nodes by degree (most connected):')
for node, degree in graph_stats['top_nodes_by_degree']:
    ntype = G.nodes[node].get('type', 'OTHER')
    print(f'  [{ntype:8s}] {node}: degree={degree}')

In [ ]:
# Visualize toàn bộ đồ thị (top 80 nodes)
print('\n📊 Đang tạo visualization đồ thị...')
visualize_graph(
    G,
    title='EV Industry Knowledge Graph — Top 80 Entities',
    top_n=80,
    save_path='output/knowledge_graph.png'
)
display(Image('output/knowledge_graph.png'))
print('✅ Đồ thị đã được lưu vào output/knowledge_graph.png')

In [ ]:
# Visualize subgraph xung quanh Tesla (2-hop)
print('\n📊 Subgraph Tesla (2-hop neighbors):')
visualize_subgraph(
    G, 
    center_node='Tesla',
    hops=2,
    title='Tesla — 2-hop Knowledge Subgraph',
    save_path='output/subgraph_Tesla.png'
)
display(Image('output/subgraph_Tesla.png'))

In [ ]:
# Visualize subgraph xung quanh BYD (2-hop)
visualize_subgraph(
    G,
    center_node='BYD',
    hops=2,
    title='BYD — 2-hop Knowledge Subgraph',
    save_path='output/subgraph_BYD.png'
)
display(Image('output/subgraph_BYD.png'))

In [ ]:
# Xem neighbors trực tiếp của Tesla trong graph
print('\n🔍 Neighbors của Tesla trong Knowledge Graph:')
print('\n  → Outgoing (Tesla là subject):')
for _, target, data in G.out_edges('Tesla', data=True):
    print(f'     Tesla --[{data["relation"]}]--> {target}')

print('\n  ← Incoming (Tesla là object):')
for source, _, data in G.in_edges('Tesla', data=True):
    print(f'     {source} --[{data["relation"]}]--> Tesla')

---
## ❓ Phần 4: Multi-hop Query Engine

**Bước 3:** Xử lý câu hỏi theo quy trình:

```
Câu hỏi → Extract Entity → Tìm Node → BFS 2-hop → Textualize → LLM Answer
```

**Ưu điểm so với Flat RAG:** Có thể kết nối thông tin từ nhiều documents qua graph traversal.

In [ ]:
# Demo GraphRAG với câu hỏi đơn
test_question = "What is the relationship between BYD and Berkshire Hathaway?"

print(f'\n❓ Câu hỏi: {test_question}')
print('\n🔍 Đang xử lý với GraphRAG...')

result = query_graphrag(
    question=test_question,
    G=G,
    client=groq_client,
    verbose=True
)

print(f'\n✅ Kết quả GraphRAG:')
print(f'   Matched nodes: {result["matched_nodes"]}')
print(f'   Nodes visited (BFS): {result["visited_nodes"]}')
print(f'   Context: {result["context_length"]} chars')
print(f'   Time: {result["time_sec"]}s')
print(f'\n💬 Câu trả lời:')
print(f'   {result["answer"]}')

print(f'\n📋 Supporting Triples (top 5):')
for t in result['supporting_triples'][:5]:
    print(f'   ({t["subject"]}) --[{t["relation"]}]--> ({t["object"]})')

In [ ]:
# Test câu hỏi multi-hop phức tạp hơn
test_questions = [
    "How did China's industrial policy lead to BYD surpassing Tesla as the world's largest EV producer?",
    "What role does CATL play in connecting Chinese battery supply chains to Western automakers?",
    "How has the Inflation Reduction Act impacted EV leasing and consumer incentives?",
]

print('\n🔍 Demo Multi-hop Queries:')
for q in test_questions:
    print(f'\n' + '='*70)
    print(f'❓ Q: {q}')
    r = query_graphrag(q, G, groq_client)
    print(f'💬 A: {r["answer"]}')
    print(f'   [{r["visited_nodes"]} nodes traversed, {r["time_sec"]}s]')

---
## 📊 Phần 5: Flat RAG Baseline (ChromaDB)

**Baseline so sánh:** Flat RAG chỉ dùng vector similarity search, không có graph traversal.

- **Embedding:** `sentence-transformers/all-MiniLM-L6-v2` (local, miễn phí)
- **Vector DB:** ChromaDB (in-memory)
- **Retrieval:** Top-5 most similar chunks

In [ ]:
# Khởi tạo và index Flat RAG
print('\n🔧 Khởi tạo Flat RAG...')
flat_rag = FlatRAG(groq_api_key=GROQ_API_KEY)

print('\n📥 Indexing tất cả chunks vào ChromaDB...')
flat_rag.index(chunks)  # ~2-5 phút tùy CPU

In [ ]:
# Test Flat RAG với cùng câu hỏi
test_question = "What is the relationship between BYD and Berkshire Hathaway?"

print(f'\n❓ Câu hỏi: {test_question}')
flat_result = flat_rag.query(test_question)

print(f'\n✅ Kết quả Flat RAG:')
print(f'   Retrieved chunks: {len(flat_result["retrieved_chunks"])}')
print(f'   Context: {flat_result["context_length"]} chars')
print(f'   Time: {flat_result["time_sec"]}s')
print(f'\n💬 Câu trả lời:')
print(f'   {flat_result["answer"]}')

print(f'\n📋 Retrieved chunks (top 3):')
for chunk in flat_result['retrieved_chunks'][:3]:
    print(f'   [Rank {chunk["rank"]}] {chunk["doc_id"]} (dist={chunk["distance"]})')
    print(f'   {chunk["text"][:150]}...')

---
## 🏆 Phần 6: Đánh Giá So Sánh — 20 Benchmark Questions

Chạy cả hai hệ thống trên **20 câu hỏi benchmark** phân loại:
- **Simple (5 câu):** Single-hop, câu hỏi về facts cụ thể
- **Multi-hop (5 câu):** Cần kết nối nhiều entities
- **Comparative (5 câu):** So sánh nhiều công ty/metrics
- **Trend (5 câu):** Phân tích xu hướng theo thời gian

In [ ]:
# Hiển thị 20 câu hỏi benchmark
print('\n📋 20 Benchmark Questions:')
for q in BENCHMARK_QUESTIONS:
    print(f'\n  Q{q["id"]:02d} [{q["category"]:12s}] {q["question"]}')

In [ ]:
# Chạy Flat RAG trên 20 câu hỏi
print('\n🔍 Chạy Flat RAG trên 20 câu hỏi...')
questions_list = [q['question'] for q in BENCHMARK_QUESTIONS]

flat_rag_results = flat_rag.batch_query(questions_list)
print(f'\n✅ FlatRAG hoàn thành {len(flat_rag_results)} câu hỏi')

In [ ]:
# Chạy GraphRAG trên 20 câu hỏi
print('\n🔍 Chạy GraphRAG trên 20 câu hỏi...')

graphrag_results = batch_query_graphrag(questions_list, G, groq_client)
print(f'\n✅ GraphRAG hoàn thành {len(graphrag_results)} câu hỏi')

In [ ]:
# Chạy evaluation
print('\n📊 Tạo bảng đánh giá...')
df_eval = run_evaluation(flat_rag_results, graphrag_results)

# In tóm tắt
print_evaluation_summary(df_eval)

# Hiển thị bảng
display(df_eval[[
    'ID', 'Category', 'Question',
    'FlatRAG_Score', 'GraphRAG_Score',
    'FlatRAG_Completeness', 'GraphRAG_Completeness',
    'FlatRAG_Time_s', 'GraphRAG_Time_s',
    'Winner'
]].style
    .background_gradient(subset=['FlatRAG_Score'], cmap='YlOrRd')
    .background_gradient(subset=['GraphRAG_Score'], cmap='YlGn')
    .set_caption('📊 Evaluation Results: FlatRAG vs GraphRAG')
)

In [ ]:
# Tạo và hiển thị biểu đồ so sánh
plot_evaluation(df_eval)
display(Image('output/evaluation_chart.png'))

In [ ]:
# Phân tích chi tiết các trường hợp GraphRAG vượt trội
print('\n🔬 Phân tích chi tiết: Những câu hỏi GraphRAG rõ ràng tốt hơn:')
graph_wins = df_eval[df_eval['GraphRAG_Score'] > df_eval['FlatRAG_Score']]

for _, row in graph_wins.iterrows():
    print(f'\n  Q{row["ID"]} [{row["Category"]}] (FlatRAG={row["FlatRAG_Score"]}, GraphRAG={row["GraphRAG_Score"]})')
    print(f'  Question: {row["Question"][:80]}...')
    print(f'  Missing in FlatRAG: {row["Missing_in_FlatRAG"]}')
    print(f'  FlatRAG  answer: {row["FlatRAG_Answer"][:150]}...')
    print(f'  GraphRAG answer: {row["GraphRAG_Answer"][:150]}...')
    print(f'  {"─" * 70}')

---
## 💰 Phần 7: Phân Tích Chi Phí (Token Usage & Time)

In [ ]:
# Ước tính chi phí indexing
total_chunks = len(chunks)
avg_chunk_words = sum(len(c['text'].split()) for c in chunks) / len(chunks)
estimated_input_tokens = total_chunks * avg_chunk_words * 1.3
estimated_output_tokens = total_chunks * 150  # ~150 tokens/triple set

# Groq hiện miễn phí với giới hạn nhất định
# Ref: https://console.groq.com/docs/rate-limits

print('\n💰 Phân Tích Chi Phí:')
print(f'  ─── INDEXING (Entity Extraction) ───')
print(f'  Số chunks xử lý: {total_chunks}')
print(f'  Avg words/chunk: {avg_chunk_words:.0f}')
print(f'  Ước tính input tokens: {estimated_input_tokens:,.0f}')
print(f'  Ước tính output tokens: {estimated_output_tokens:,.0f}')
print(f'  Groq (llama-3.3-70b): $0.00 (free tier, ~6K TPM limit)')
print(f'  OpenAI GPT-4o-mini (nếu dùng): ~${(estimated_input_tokens * 0.15 + estimated_output_tokens * 0.6) / 1e6:.3f} USD')

# Query cost
total_query_flat   = sum(r['time_sec'] for r in flat_rag_results)
total_query_graph  = sum(r['time_sec'] for r in graphrag_results)
total_context_flat = sum(r['context_length'] for r in flat_rag_results)
total_context_graph = sum(r['context_length'] for r in graphrag_results)

print(f'\n  ─── QUERYING (20 câu hỏi) ───')
print(f'  FlatRAG total time: {total_query_flat:.1f}s (avg {total_query_flat/20:.1f}s/query)')
print(f'  GraphRAG total time: {total_query_graph:.1f}s (avg {total_query_graph/20:.1f}s/query)')
print(f'  FlatRAG avg context: {total_context_flat//20:,} chars')
print(f'  GraphRAG avg context: {total_context_graph//20:,} chars')

print(f'\n  ─── KẾT LUẬN ───')
print(f'  • GraphRAG có chi phí indexing cao hơn (phải gọi LLM để extract triples)')
print(f'  • Nhưng sau khi có graph, query có thể nhanh hơn cho multi-hop questions')
print(f'  • Graph được cache → chỉ cần index 1 lần, query nhiều lần')
print(f'  • Groq free tier đủ dùng cho lab này (không tốn chi phí thực tế)')

---
## 📝 Phần 8: Kết Luận & Báo Cáo

In [ ]:
# Tóm tắt so sánh cuối cùng
flat_avg  = df_eval['FlatRAG_Score'].mean()
graph_avg = df_eval['GraphRAG_Score'].mean()
winner_counts = df_eval['Winner'].value_counts().to_dict()

summary_html = f"""
<div style="font-family: monospace; background: #0D1117; color: #E6EDF3; padding: 20px; border-radius: 8px;">
<h2 style="color: #58A6FF;">📊 BÁO CÁO CUỐI: GraphRAG vs Flat RAG</h2>

<table style="width:100%; border-collapse: collapse;">
  <tr style="background:#161B22;">
    <th style="padding:8px; border:1px solid #30363D; color:#58A6FF;">Tiêu chí</th>
    <th style="padding:8px; border:1px solid #30363D; color:#58A6FF;">Flat RAG</th>
    <th style="padding:8px; border:1px solid #30363D; color:#F97316;">GraphRAG</th>
  </tr>
  <tr>
    <td style="padding:8px; border:1px solid #30363D;">Avg Score (20 câu)</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">{flat_avg:.2f}/10</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">{graph_avg:.2f}/10</td>
  </tr>
  <tr style="background:#161B22;">
    <td style="padding:8px; border:1px solid #30363D;">Số câu thắng</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">{winner_counts.get('FlatRAG', 0)}</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">{winner_counts.get('GraphRAG', 0)}</td>
  </tr>
  <tr>
    <td style="padding:8px; border:1px solid #30363D;">Multi-hop accuracy</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">⚠️ Thấp</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">✅ Cao hơn</td>
  </tr>
  <tr style="background:#161B22;">
    <td style="padding:8px; border:1px solid #30363D;">Indexing cost</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">🟢 Thấp</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">🟡 Cao hơn (LLM call)</td>
  </tr>
  <tr>
    <td style="padding:8px; border:1px solid #30363D;">Explainability</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">⚠️ Black box</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">✅ Reasoning path visible</td>
  </tr>
  <tr style="background:#161B22;">
    <td style="padding:8px; border:1px solid #30363D;">Hallucination risk</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">🔴 Cao hơn</td>
    <td style="padding:8px; border:1px solid #30363D; text-align:center;">🟢 Thấp hơn (grounded)</td>
  </tr>
</table>

<h3 style="color:#58A6FF; margin-top:20px;">🔍 Kết luận:</h3>
<p>GraphRAG vượt trội với các câu hỏi <strong>Multi-hop</strong> và <strong>Comparative</strong> — những câu cần kết nối thông tin từ nhiều nguồn. 
Flat RAG đủ dùng cho câu hỏi đơn giản nhưng dễ bị ảo giác khi cần reasoning phức tạp.</p>
</div>
"""

display(HTML(summary_html))

In [ ]:
# Lưu kết quả cuối cùng
import json
from pathlib import Path

final_results = {
    'metadata': {
        'student': 'Nguyen Quang Anh',
        'student_id': '2A202600608',
        'lab': 'Day 19 - GraphRAG',
        'dataset': '70 EV Corpus documents',
        'llm': 'Groq llama-3.3-70b-versatile',
        'embedding_model': 'all-MiniLM-L6-v2',
    },
    'graph_stats': {
        'num_nodes': G.number_of_nodes(),
        'num_edges': G.number_of_edges(),
        'num_triples_extracted': len(triples),
    },
    'evaluation_summary': {
        'flat_rag_avg_score': round(df_eval['FlatRAG_Score'].mean(), 2),
        'graphrag_avg_score': round(df_eval['GraphRAG_Score'].mean(), 2),
        'winner_counts': df_eval['Winner'].value_counts().to_dict(),
        'flat_rag_avg_time': round(df_eval['FlatRAG_Time_s'].mean(), 2),
        'graphrag_avg_time': round(df_eval['GraphRAG_Time_s'].mean(), 2),
    },
}

Path('output').mkdir(exist_ok=True)
with open('output/final_report.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print('\n✅ Đã lưu báo cáo cuối vào output/final_report.json')
print('\n📁 Files đã tạo:')
for f in Path('output').iterdir():
    size = f.stat().st_size
    print(f'   {f.name}: {size:,} bytes')